<a href="https://colab.research.google.com/github/t114c75034/AI-Security_0518-HW/blob/main/%E3%80%8CCNN_1_ipynb%E3%80%8D_114C75034.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from tensorflow import keras
from tensorflow.keras import layers
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        11,530 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 104,202 (407.04 KB)

 Trainable params: 104,202 (407.04 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype("float32") / 255
model.compile(optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
model.fit(train_images, train_labels, epochs=5, batch_size=128,validation_split=0.2)
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test accuracy: {test_acc:.3f}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 44s 114ms/step - accuracy: 0.9190 - loss: 0.2588 - val_accuracy: 0.9542 - val_loss: 0.1557
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 44s 118ms/step - accuracy: 0.9811 - loss: 0.0604 - val_accuracy: 0.9837 - val_loss: 0.0523
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 43s 114ms/step - accuracy: 0.9874 - loss: 0.0402 - val_accuracy: 0.9880 - val_loss: 0.0450
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 42s 111ms/step - accuracy: 0.9908 - loss: 0.0292 - val_accuracy: 0.9882 - val_loss: 0.0388
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 83s 114ms/step - accuracy: 0.9930 - loss: 0.0224 - val_accuracy: 0.9885 - val_loss: 0.0399
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9896 - loss: 0.0282
Test accuracy: 0.990


## PyTorch Implementation

Below is the equivalent implementation of the CNN model and its training process using PyTorch.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

### Define the PyTorch Model

This `ConvNet` class mirrors the architecture of your Keras model, using `nn.Conv2d`, `nn.MaxPool2d`, `nn.Flatten`, and `nn.Linear` layers.

In [4]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)
        # Calculate the size after convolutional layers to determine input for the first dense layer
        # MNIST images are 28x28. After 3x3 conv, 2x2 max pool, 3x3 conv, 2x2 max pool, 3x3 conv:
        # 28 -> (28-3)+1 = 26 -> 26/2 = 13
        # 13 -> (13-3)+1 = 11 -> 11/2 = 5 (integer division)
        # 5 -> (5-3)+1 = 3
        self.fc1 = nn.Linear(128 * 3 * 3, 10) # 128 filters, 3x3 output size

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = x.view(-1, 128 * 3 * 3) # Flatten the output for the fully connected layer
        x = self.fc1(x)
        return F.log_softmax(x, dim=1)

model_pt = ConvNet()
print(model_pt)

# Optional: Check if CUDA is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pt.to(device)

ConvNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=1152, out_features=10, bias=True)
)


ConvNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=1152, out_features=10, bias=True)
)

### Load and Preprocess Data

We'll use `torchvision.datasets.MNIST` to load the dataset and `transforms` to convert images to tensors and normalize them. `DataLoader`s are then created for efficient batching.

In [5]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Mean and std deviation for MNIST
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create DataLoaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 486kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.43MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.66MB/s]


### Define Loss Function and Optimizer

`nn.NLLLoss` is chosen because our model outputs `log_softmax`, which is common in PyTorch for classification. `torch.optim.RMSprop` is used for consistency with the Keras example.

In [6]:
criterion = nn.NLLLoss()
optimizer = torch.optim.RMSprop(model_pt.parameters(), lr=0.001) # Learning rate can be tuned

### Training and Evaluation Loop

PyTorch requires a manual training loop. Here, we iterate through epochs, then through batches, perform forward and backward passes, and update weights. Validation is performed after each epoch.

In [9]:
def train(model, device, train_loader, optimizer, epoch):
    model.train() # Set model to training mode
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # Zero the gradients before each backward pass
        output = model(data)
        loss = criterion(output, target)
        loss.backward() # Compute gradient of the loss with respect to model parameters
        optimizer.step() # Update model parameters
        running_loss += loss.item() * data.size(0) # Sum up batch loss, multiplied by batch size
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')
    avg_train_loss = running_loss / len(train_loader.dataset)
    return avg_train_loss

def test(model, device, test_loader):
    model.eval() # Set model to evaluation mode
    test_loss = 0
    correct = 0
    with torch.no_grad(): # Disable gradient calculation during evaluation
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0) # Sum up batch loss, multiplied by batch size
            pred = output.argmax(dim=1, keepdim=True) # Get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset) # Average loss per sample
    test_acc = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({test_acc:.0f}%)\n')
    return test_loss, test_acc

In [10]:
epochs = 5
train_losses_pt = []
test_losses_pt = []
test_accuracies_pt = []

for epoch in range(1, epochs + 1):
    avg_train_loss = train(model_pt, device, train_loader, optimizer, epoch)
    test_loss_val, test_acc_val = test(model_pt, device, test_loader)
    train_losses_pt.append(avg_train_loss)
    test_losses_pt.append(test_loss_val)
    test_accuracies_pt.append(test_acc_val)
    print(f"Epoch {epoch}: Train Loss: {avg_train_loss:.4f}, Test Loss: {test_loss_val:.4f}, Test Acc: {test_acc_val:.2f}%")

print(f"PyTorch Final Test accuracy: {test_accuracies_pt[-1]:.3f}%")

Train Epoch: 1 [0/60000 (0%)]	Loss: 0.013890
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.007573
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.018511
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.008035
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.002863

Test set: Average loss: 0.0282, Accuracy: 9913/10000 (99%)

Epoch 1: Train Loss: 0.0147, Test Loss: 0.0282, Test Acc: 99.13%
Train Epoch: 2 [0/60000 (0%)]	Loss: 0.008631
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.001636
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.001573
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.019049
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.006101

Test set: Average loss: 0.0258, Accuracy: 9923/10000 (99%)

Epoch 2: Train Loss: 0.0123, Test Loss: 0.0258, Test Acc: 99.23%
Train Epoch: 3 [0/60000 (0%)]	Loss: 0.013601
Train Epoch: 3 [12800/60000 (21%)]	Loss: 0.000990
Train Epoch: 3 [25600/60000 (43%)]	Loss: 0.011287
Train Epoch: 3 [38400/60000 (64%)]	Loss: 0.001420
Train Epoch: 3 [51200/60000 (85%)]	Loss: 0.000960

Test set: Av

### Visualize Loss Curves

Let's plot the training and validation loss for both the TensorFlow/Keras and PyTorch models to compare their learning curves.

In [ ]:
import matplotlib.pyplot as plt

# Extract TensorFlow/Keras metrics from the previous output
keras_train_loss = [0.2588, 0.0604, 0.0402, 0.0292, 0.0224]
keras_val_loss = [0.1557, 0.0523, 0.0450, 0.0388, 0.0399]

epochs_list = range(1, epochs + 1)

plt.figure(figsize=(12, 6))

# Plot TensorFlow/Keras Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_list, keras_train_loss, label='Keras Train Loss')
plt.plot(epochs_list, keras_val_loss, label='Keras Validation Loss')
plt.title('TensorFlow/Keras Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot PyTorch Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_list, train_losses_pt, label='PyTorch Train Loss')
plt.plot(epochs_list, test_losses_pt, label='PyTorch Test Loss')
plt.title('PyTorch Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()